# Preprocessing

In [72]:
import pandas as pd
import seaborn as sns
import datetime

from sklearn.model_selection import train_test_split
from sklearn.linear_model import ridge_regression
from sklearn.preprocessing import StandardScaler

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
path = "/content/drive/MyDrive/sp-26/ds4400/project/NewVLRDataRaw.csv"

df_vlr_raw = pd.read_csv(path)
df_vlr_raw.info()
df_vlr_raw.head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 129156 entries, 0 to 129155
Data columns (total 67 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   MatchID             129156 non-null  int64  
 1   GameID              129156 non-null  int64  
 2   EventID             129156 non-null  int64  
 3   Date                129156 non-null  object 
 4   Team1ID             129156 non-null  int64  
 5   Team2ID             129156 non-null  int64  
 6   Team1 Name          129152 non-null  object 
 7   Team2 Name          129155 non-null  object 
 8   Series Odds         129156 non-null  float64
 9   Team1 Map Odds      129156 non-null  float64
 10  Map                 129156 non-null  int64  
 11  Team1 Rounds        129156 non-null  int64  
 12  Team2 Rounds        129156 non-null  int64  
 13  Team1 Atk Rounds    129151 non-null  float64
 14  Team2 Atk Rounds    129152 non-null  float64
 15  Team1 Def Rounds    129154 non-nul

,MatchID,GameID,EventID,Date,Team1ID,Team2ID,Team1 Name,Team2 Name,Series Odds,Team1 Map Odds,...,Team1Game2,Team1Game3,Team1Game4,Team1Game5,Team2Game1,Team2Game2,Team2Game3,Team2Game4,Team2Game5,VOD Link
0,381258,180991,2158,2024-09-04,15139,15138,MYVRA,KS Hunters,0.00,0.000000,...,381252,381247,381242,365378,381253,381251,381249,381243,365377,https://youtu.be/-iaPdT6K73o?t=974
1,381258,180992,2158,2024-09-04,15139,15138,MYVRA,KS Hunters,0.00,0.000000,...,381252,381247,381242,365378,381253,381251,381249,381243,365377,https://youtu.be/NmoNwQvqT0U?t=436
2,381287,181075,2159,2024-09-04,7413,9205,FiRePOWER,LEVIATÁN GC,2.40,2.251173,...,381279,381277,381272,365392,381284,381278,381275,381273,326896,https://youtu.be/cJO-ueXNi0I?t=1303
3,381287,181076,2159,2024-09-04,7413,9205,FiRePOWER,LEVIATÁN GC,2.40,2.251173,...,381279,381277,381272,365392,381284,381278,381275,381273,326896,https://youtu.be/ljYModJBqFU?t=470
4,381287,181077,2159,2024-09-04,7413,9205,FiRePOWER,LEVIATÁN GC,2.40,2.251173,...,381279,381277,381272,365392,381284,381278,381275,381273,326896,https://youtu.be/W_7Oq_L7iv8?t=384
5,398800,183415,2143,2024-09-04,11479,14461,Apeks,Barça eSports,1.13,1.267860,...,398786,398782,370336,370335,398794,398790,398788,363294,363293,https://youtu.be/Li-sxK6NEhI?list=PLTlFZO3kdyG...
6,398800,183416,2143,2024-09-04,11479,14461,Apeks,Barça eSports,1.13,1.267860,...,398786,398782,370336,370335,398794,398790,398788,363294,363293,https://youtu.be/0rDczTqlBeQ?list=PLTlFZO3kdyG...
7,398800,183417,2143,2024-09-04,11479,14461,Apeks,Barça eSports,1.13,1.267860,...,398786,398782,370336,370335,398794,398790,398788,363294,363293,https://youtu.be/xBhF70h6Xgc?list=PLTlFZO3kdyG...
8,398801,183418,2143,2024-09-04,8044,11232,MOUZ,Joblife,2.70,2.423109,...,398787,398783,367234,367232,398795,398791,398789,349276,349274,https://youtu.be/d6G6QUhHWE4?list=PLTlFZO3kdyG...
9,398801,183419,2143,2024-09-04,8044,11232,MOUZ,Joblife,2.70,2.423109,...,398787,398783,367234,367232,398795,398791,398789,349276,349274,https://youtu.be/CUt9cHF23uE?list=PLTlFZO3kdyG...


In [56]:
# observe categorical features

print(df_vlr_raw.select_dtypes(include='object').columns.to_list())

df_vlr_raw.select_dtypes(include='object')

['Date', 'Team1 Name', 'Team2 Name', 'Round Breakdown', 'VOD Link']


,Date,Team1 Name,Team2 Name,Round Breakdown,VOD Link
0,2024-09-04,MYVRA,KS Hunters,0100!0.6!0.1!0120!0.9!9.7!1023!13.3!1.1!1033!2...,https://youtu.be/-iaPdT6K73o?t=974
1,2024-09-04,MYVRA,KS Hunters,0000!0.4!0.3!0020!2.8!6.8!0022!14.1!0.6!0031!1...,https://youtu.be/NmoNwQvqT0U?t=436
2,2024-09-04,FiRePOWER,LEVIATÁN GC,0100!0.1!0.2!0122!0.1!0.8!1021!5.7!7.9!1013!9....,https://youtu.be/cJO-ueXNi0I?t=1303
3,2024-09-04,FiRePOWER,LEVIATÁN GC,0000!0.3!0.2!0020!4.7!10.2!0023!15.7!0.9!0032!...,https://youtu.be/ljYModJBqFU?t=470
4,2024-09-04,FiRePOWER,LEVIATÁN GC,1100!0.2!0.3!1102!9.6!0.7!0031!0.9!11.4!1133!7...,https://youtu.be/W_7Oq_L7iv8?t=384
...,...,...,...,...,...
129151,2024-08-09,Ball Knowers,Scarlet Knights Black,0100!0.2!0.3!1020!4.3!7.6!1013!9.2!8.7!1033!1....,NaN
129152,2024-08-09,Ball Knowers,Scarlet Knights Black,0100!0.2!0.2!0120!1.8!7.2!1023!11.7!0.6!0133!1...,NaN
129153,2022-10-24,Gladiskal,Stryx,0000!0.2!0.1!0020!2.0!9.1!1123!14.7!0.5!0032!4...,NaN
129154,2022-10-24,Gladiskal,Stryx,0000!0.1!0.3!0020!1.1!8.2!0023!11.4!0.6!0031!1...,NaN


# Categorical Variables
Of the categorical variables, we can assume that the Team Names do not affect because an infinite number of names can be made and it is unordered. This feature is also redundant because there are Team1ID and Team2ID. We can also assume that the VOD Link will not affect our prediction because it is completely randomly generated.

## Date
Our dates range from 2020 to 2024, so we have a la|rge number of dates that can tell us about how win conditions may change over time. However, for our purposes date will not influence our outcome as much as other numerical features and potentially add noise. Therefore, we will drop it

## Round Breakdown
This should be removed because it tells us who wins which round and directly tells us which team will win if we extract the round information from the round breakdown.

In [79]:
categorical_drop = ['Team1 Name', 'Team2 Name', 'Date', 'Round Breakdown', 'VOD Link']
df_vlr_clean = df_vlr_raw.drop(columns=categorical_drop)


df_vlr_clean.head(10)

,MatchID,GameID,EventID,Team1ID,Team2ID,Series Odds,Team1 Map Odds,Map,Team1 Rounds,Team2 Rounds,...,Team1Game1,Team1Game2,Team1Game3,Team1Game4,Team1Game5,Team2Game1,Team2Game2,Team2Game3,Team2Game4,Team2Game5
0,381258,180991,2158,15139,15138,0.00,0.000000,11,13,9,...,381254,381252,381247,381242,365378,381253,381251,381249,381243,365377
1,381258,180992,2158,15139,15138,0.00,0.000000,1,13,10,...,381254,381252,381247,381242,365378,381253,381251,381249,381243,365377
2,381287,181075,2159,7413,9205,2.40,2.251173,11,4,13,...,381282,381279,381277,381272,365392,381284,381278,381275,381273,326896
3,381287,181076,2159,7413,9205,2.40,2.251173,3,13,6,...,381282,381279,381277,381272,365392,381284,381278,381275,381273,326896
4,381287,181077,2159,7413,9205,2.40,2.251173,9,13,10,...,381282,381279,381277,381272,365392,381284,381278,381275,381273,326896
5,398800,183415,2143,11479,14461,1.13,1.267860,10,10,13,...,398792,398786,398782,370336,370335,398794,398790,398788,363294,363293
6,398800,183416,2143,11479,14461,1.13,1.267860,11,13,5,...,398792,398786,398782,370336,370335,398794,398790,398788,363294,363293
7,398800,183417,2143,11479,14461,1.13,1.267860,3,13,11,...,398792,398786,398782,370336,370335,398794,398790,398788,363294,363293
8,398801,183418,2143,8044,11232,2.70,2.423109,9,13,10,...,398793,398787,398783,367234,367232,398795,398791,398789,349276,349274
9,398801,183419,2143,8044,11232,2.70,2.423109,2,7,13,...,398793,398787,398783,367234,367232,398795,398791,398789,349276,349274


In [81]:
df_vlr_clean.loc[0]

,0
MatchID,381258.0
GameID,180991.0
EventID,2158.0
Team1ID,15139.0
Team2ID,15138.0
...,...
Team2Game1,381253.0
Team2Game2,381251.0
Team2Game3,381249.0
Team2Game4,381243.0


## Game/Match ID and Team ID
Both of these IDs are represented as integers, but count as categorical data. Neither of them are useful because they seem to be randomly generated to some extent. Therefore, we will drop any ID column including the Team1Game1 type columns

In [82]:
print(df_vlr_clean.columns)

id_columns = ['MatchID', 'GameID', 'EventID', 'Team1ID', 'Team2ID', 'Team1Game1', 'Team1Game2', 'Team1Game3',
       'Team1Game4', 'Team1Game5', 'Team2Game1', 'Team2Game2', 'Team2Game3',
       'Team2Game4', 'Team2Game5']

df_vlr_clean = df_vlr_clean.drop(columns=id_columns)
df_vlr_clean.head(10)

Index(['MatchID', 'GameID', 'EventID', 'Team1ID', 'Team2ID', 'Series Odds',
       'Team1 Map Odds', 'Map', 'Team1 Rounds', 'Team2 Rounds',
       'Team1 Atk Rounds', 'Team2 Atk Rounds', 'Team1 Def Rounds',
       'Team2 Def Rounds', 'Team1 Rating', 'Team2 Rating', 'Team1 ACS',
       'Team2 ACS', 'Team1 Kills', 'Team2 Kills', 'Team1 Deaths',
       'Team2 Deaths', 'Team1 Assists', 'Team2 Assists', 'Team1 DeltaK/D',
       'Team2 DeltaK/D', 'Team1 KAST', 'Team2 KAST', 'Team1 ADR', 'Team2 ADR',
       'Team1 HS', 'Team2 HS', 'Team1 FK', 'Team2 FK', 'Team1 FD', 'Team2 FD',
       'Team1 DeltaFK/FD', 'Team2 DeltaFK/FD', 'Team1 Pistols',
       'Team2 Pistols', 'Team1 EcosWon', 'Team2 EcosWon', 'Team1 Ecos',
       'Team2 Ecos', 'Team1 Semibuys Won', 'Team2 Semibuys Won',
       'Team1 Semibuys', 'Team2 Semibuys', 'Team1 Fullbuys Won',
       'Team2 Fullbuys Won', 'Team1 Fullbuys', 'Team2 Fullbuys', 'Team1Game1',
       'Team1Game2', 'Team1Game3', 'Team1Game4', 'Team1Game5', 'Team2Game1',


,Series Odds,Team1 Map Odds,Map,Team1 Rounds,Team2 Rounds,Team1 Atk Rounds,Team2 Atk Rounds,Team1 Def Rounds,Team2 Def Rounds,Team1 Rating,...,Team1 Ecos,Team2 Ecos,Team1 Semibuys Won,Team2 Semibuys Won,Team1 Semibuys,Team2 Semibuys,Team1 Fullbuys Won,Team2 Fullbuys Won,Team1 Fullbuys,Team2 Fullbuys
0,0.00,0.000000,11,13,9,7.0,6.0,6.0,3.0,1.134,...,2.0,3.0,4.0,2.0,7.0,4.0,7.0,7.0,11.0,13.0
1,0.00,0.000000,1,13,10,4.0,2.0,9.0,8.0,1.066,...,2.0,4.0,2.0,0.0,5.0,5.0,8.0,8.0,14.0,12.0
2,2.40,2.251173,11,4,13,1.0,9.0,3.0,4.0,0.628,...,3.0,1.0,2.0,1.0,7.0,3.0,1.0,10.0,5.0,11.0
3,2.40,2.251173,3,13,6,11.0,5.0,2.0,1.0,1.174,...,2.0,2.0,3.0,2.0,3.0,7.0,9.0,3.0,12.0,8.0
4,2.40,2.251173,9,13,10,5.0,3.0,8.0,7.0,1.102,...,1.0,5.0,3.0,2.0,4.0,6.0,9.0,7.0,16.0,10.0
5,1.13,1.267860,10,10,13,9.0,10.0,1.0,3.0,0.888,...,4.0,2.0,3.0,2.0,5.0,6.0,6.0,9.0,12.0,13.0
6,1.13,1.267860,11,13,5,6.0,5.0,7.0,0.0,1.280,...,1.0,4.0,3.0,1.0,3.0,3.0,8.0,4.0,12.0,9.0
7,1.13,1.267860,3,13,11,2.0,1.0,11.0,10.0,1.172,...,3.0,1.0,1.0,2.0,3.0,9.0,11.0,8.0,16.0,12.0
8,2.70,2.423109,9,13,10,6.0,5.0,7.0,5.0,1.084,...,0.0,3.0,5.0,1.0,9.0,3.0,6.0,9.0,12.0,15.0
9,2.70,2.423109,2,7,13,1.0,6.0,6.0,7.0,0.814,...,3.0,0.0,1.0,4.0,3.0,7.0,4.0,8.0,12.0,11.0


# Target Feature
The dataset itself doesn't include a target feature, so we can use the Team1 rounds and Team2 rounds features to decide if Team1 won the game. We will also check to see if there are any ties in the data 

## Invalid Round Scores
In a standard VLR match, the winner must have won at least 13 rounds. Therefore, entries are invalid if at least one team in the row does not have at least 13 rounds won.

We will drop these games because they will not help us train our models.

Once we make the new feature, we will drop the two rounds features.

In [83]:
# remove rows with invalid round scores
df_vlr_clean = df_vlr_clean[(df_vlr_clean['Team1 Rounds'] >= 13) | (df_vlr_clean['Team2 Rounds'] >= 13)]

# remove draws because doesn't help us determine if Team 1 beat Team 2
df_vlr_clean = df_vlr_clean[(df_vlr_clean['Team1 Rounds'] != df_vlr_clean['Team2 Rounds'])]

# create new column for Team1 Win
df_vlr_clean['Team1 Win'] = (df_vlr_clean['Team1 Rounds'] > df_vlr_clean['Team2 Rounds']).astype(int)

# drop Team1 Rounds and Team2 Rounds
df_vlr_clean = df_vlr_clean.drop(columns=['Team1 Rounds', 'Team2 Rounds'])
df_vlr_clean.head(10)

,Series Odds,Team1 Map Odds,Map,Team1 Atk Rounds,Team2 Atk Rounds,Team1 Def Rounds,Team2 Def Rounds,Team1 Rating,Team2 Rating,Team1 ACS,...,Team2 Ecos,Team1 Semibuys Won,Team2 Semibuys Won,Team1 Semibuys,Team2 Semibuys,Team1 Fullbuys Won,Team2 Fullbuys Won,Team1 Fullbuys,Team2 Fullbuys,Team1 Win
0,0.00,0.000000,11,7.0,6.0,6.0,3.0,1.134,0.870,211.4,...,3.0,4.0,2.0,7.0,4.0,7.0,7.0,11.0,13.0,1
1,0.00,0.000000,1,4.0,2.0,9.0,8.0,1.066,0.950,211.2,...,4.0,2.0,0.0,5.0,5.0,8.0,8.0,14.0,12.0,1
2,2.40,2.251173,11,1.0,9.0,3.0,4.0,0.628,1.284,150.2,...,1.0,2.0,1.0,7.0,3.0,1.0,10.0,5.0,11.0,0
3,2.40,2.251173,3,11.0,5.0,2.0,1.0,1.174,0.826,206.4,...,2.0,3.0,2.0,3.0,7.0,9.0,3.0,12.0,8.0,1
4,2.40,2.251173,9,5.0,3.0,8.0,7.0,1.102,0.924,224.0,...,5.0,3.0,2.0,4.0,6.0,9.0,7.0,16.0,10.0,1
5,1.13,1.267860,10,9.0,10.0,1.0,3.0,0.888,1.102,187.0,...,2.0,3.0,2.0,5.0,6.0,6.0,9.0,12.0,13.0,0
6,1.13,1.267860,11,6.0,5.0,7.0,0.0,1.280,0.680,223.0,...,4.0,3.0,1.0,3.0,3.0,8.0,4.0,12.0,9.0,1
7,1.13,1.267860,3,2.0,1.0,11.0,10.0,1.172,0.818,208.6,...,1.0,1.0,2.0,3.0,9.0,11.0,8.0,16.0,12.0,1
8,2.70,2.423109,9,6.0,5.0,7.0,5.0,1.084,0.904,211.2,...,3.0,5.0,1.0,9.0,3.0,6.0,9.0,12.0,15.0,1
9,2.70,2.423109,2,1.0,6.0,6.0,7.0,0.814,1.172,196.6,...,0.0,1.0,4.0,3.0,7.0,4.0,8.0,12.0,11.0,0


In [ ]:
# check for missing values
df_vlr_clean.isnull().sum()

,0
Series Odds,0
Team1 Map Odds,0
Map,0
Team1 Atk Rounds,5
Team2 Atk Rounds,4
Team1 Def Rounds,2
Team2 Def Rounds,5
Team1 Rating,0
Team2 Rating,0
Team1 ACS,0


## Other Round Data
There are other features that can be directly used to calculate the number of rounds won on each side. We don't want to model to know how many rounds either team has because if we knew how many rounds either team had, our outcomes would be trivial. Knowing how many rounds were won on each side directly tells you who won. We will have to drop these features.


In [85]:
round_features = ['Team1 Atk Rounds', 'Team2 Atk Rounds', 'Team1 Def Rounds',
                  'Team2 Def Rounds', 'Team1 EcosWon', 'Team2 EcosWon', 'Team1 Semibuys Won',
                  'Team2 Semibuys Won', 'Team1 Fullbuys Won', 'Team2 Fullbuys Won']

df_vlr_clean.drop(columns=round_features)

,Series Odds,Team1 Map Odds,Map,Team1 Rating,Team2 Rating,Team1 ACS,Team2 ACS,Team1 Kills,Team2 Kills,Team1 Deaths,...,Team2 DeltaFK/FD,Team1 Pistols,Team2 Pistols,Team1 Ecos,Team2 Ecos,Team1 Semibuys,Team2 Semibuys,Team1 Fullbuys,Team2 Fullbuys,Team1 Win
0,0.0,0.000000,11,1.134,0.870,211.4,178.2,83.0,70.0,70.0,...,-2.0,2.0,0.0,2.0,3.0,7.0,4.0,11.0,13.0,1
1,0.0,0.000000,1,1.066,0.950,211.2,198.2,87.0,82.0,82.0,...,1.0,1.0,1.0,2.0,4.0,5.0,5.0,14.0,12.0,1
2,2.4,2.251173,11,0.628,1.284,150.2,216.2,41.0,70.0,72.0,...,3.0,1.0,1.0,3.0,1.0,7.0,3.0,5.0,11.0,0
3,2.4,2.251173,3,1.174,0.826,206.4,173.0,69.0,56.0,56.0,...,5.0,1.0,1.0,2.0,2.0,3.0,7.0,12.0,8.0,1
4,2.4,2.251173,9,1.102,0.924,224.0,196.4,92.0,77.0,77.0,...,-1.0,1.0,1.0,1.0,5.0,4.0,6.0,16.0,10.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
129150,0.0,0.000000,3,0.922,1.114,199.6,212.8,81.0,88.0,88.0,...,0.0,1.0,1.0,2.0,3.0,5.0,3.0,15.0,16.0,0
129151,0.0,0.000000,5,1.132,0.854,213.0,188.0,94.0,81.0,81.0,...,-10.0,2.0,0.0,3.0,3.0,5.0,6.0,14.0,13.0,1
129152,0.0,0.000000,1,1.026,1.024,207.6,206.2,129.0,128.0,128.0,...,-2.0,1.0,1.0,2.0,2.0,7.0,5.0,25.0,27.0,1
129153,0.0,0.000000,6,1.534,0.450,257.8,123.2,68.0,26.0,26.0,...,-6.0,2.0,0.0,0.0,3.0,3.0,6.0,9.0,3.0,1


# Economy information
The last few columns for pistols, ecos semibuys, and fullbuys has many empty values, but some values are missing. There is a good chance that certain economy strategies find more success than others, so we will fill missing values with dummy values instead of fully dropping the rows.

## Ecos Won, Semibuys Won, Fullbuys Won
These columns will have to be removed because they have the same issue 